In [1]:
# Install if needed:
# !pip install pandas scikit-learn nltk matplotlib

import pandas as pd
import re
import nltk
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

nltk.download("stopwords")

# 1. Load dataset
df = pd.read_csv("QH_Dataset.csv")
print(df.head())
print(df.columns)

# 2. Select text column
text_col = df.select_dtypes(include="object").columns[0]
print("Text column:", text_col)

# 3. NLP preprocessing
stop_words = set(stopwords.words("english"))

def clean(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", "", text)
    return " ".join(w for w in text.split() if w not in stop_words)

df["clean_text"] = df[text_col].fillna("").apply(clean)

# 4. TF-IDF
tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(df["clean_text"])

# 5. K-Means
k = 5
model = KMeans(n_clusters=k, random_state=42, n_init=10)
df["Cluster"] = model.fit_predict(X)

# 6. Show results
print("\nCluster Counts:")
print(df["Cluster"].value_counts().sort_index())

print("\nSample Results:")
print(df[[text_col, "Cluster"]].head(20))

# 7. Show important words in each cluster
words = tfidf.get_feature_names_out()

for i in range(k):
    top = model.cluster_centers_[i].argsort()[-10:][::-1]
    print(f"\nCluster {i}:", ", ".join(words[top]))

# 8. Save results
df.to_csv("QH_Dataset_Clustered.csv", index=False)

print("\nDone! Saved as QH_Dataset_Clustered.csv")

                                               Verse  \
0                           والذين هم لفروجهم حافظون   
1  وإن تطع أكثر من في الأرض يضلوك عن سبيل الله إن...   
2  ذلك هدى الله يهدي به من يشاء من عباده ولو أشرك...   
3  وأتموا الحج والعمرة لله فإن أحصرتم فما استيسر ...   
4  ولن تستطيعوا أن تعدلوا بين النساء ولو حرصتم فل...   

                                              Hadith  Label  
0  سمعت رسول الله صلى الله عليه وسلم يقول من استط...      1  
1  قال رسول الله صلى الله عليه وسلم نفس المؤمن مع...      0  
2  أن رسول الله صلى الله عليه وسلم ركب فرسا فصرع ...      0  
3  رسول الله صلى الله عليه وسلم فقالت إني امرأة ث...      1  
4  قال سمعت النبي صلى الله عليه وسلم يقول قال الل...      0  
Index(['Verse', 'Hadith', 'Label'], dtype='str')
Text column: Verse


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
/var/folders/88/w4w1n8l12kd_z42_6mrnndmw0000gn/T/ipykernel_4387/3676613865.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_col = df.select_dtypes(include="object").columns[0]


ValueError: empty vocabulary; perhaps the documents only contain stop words